# Explore the macroalgal microbiome use case

This notebook analyzes MAGs for the macroalgal microbiome use case.

**Inputs**: data in `../data/marine-use-case/data/`:
- `drep.csv`: dRep clustering output
- `checkm2.tsv`: CheckM2 quality assessment
- `gtdb.tsv`: GTDB taxonomy classification
- `quast.tsv`: QUAST assembly statistics
- `bakta.tsv`: BAKTA genome annotation

**Outputs**
- Plots: `../results/marine-use-case/`

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

sys.path.insert(0, str((Path.cwd() / "bin").resolve()))
sys.path.insert(0, str(Path.cwd().resolve()))

from helpers import (
    load_dfs,
    compute_print_stats,
    explore_species_level_clusters_all,
    compute_taxo_classification_summary,
    get_all_taxo_levels,
    get_bakta_annot_df,
    get_kegg_path_df,
    get_relative_abund_taxo_levels,
    print_stats,
)

In [2]:
uc_name = "macroalgal-epiphytic"

metadata_df, reps_df, coverage_df = load_dfs(uc_name)

In [3]:
# WARNING: high percentage of unmapped
print("Percentage of unmapped reads for coverage")
print_stats(coverage_df.query("Genome == 'unmapped'").drop(columns="Genome").T.describe())

Percentage of unmapped reads for coverage
585: 54.92 ± 16.42, Median: 50.71, IQR: 45.86-61.87, Range: 41.01-73.04


# Quality

In [4]:
print(f"Total number of MAGs: {reps_df['Cluster members'].sum()}")
print(f"Total number of species-level clusters: {reps_df.shape[0]}")

Total number of MAGs: 897
Total number of species-level clusters: 832


In [5]:
explore_species_level_clusters_all(reps_df)

Species-level clusters with no contamination threshold
Total number: 832.0
Cluster members: 1.08 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.42 ± 1.95, Median: 0.62, IQR: 0.06-1.91, Range: 0.00-10.18
Completeness: 44.39 ± 36.55, Median: 30.80, IQR: 9.70-88.40, Range: 0.00-100.00
Total length: 1.82 ± 1.61, Median: 1.28, IQR: 0.40-2.96, Range: 0.05-7.59

Species-level clusters with contamination < 5%
Total number: 771.0
Cluster members: 1.08 ± 0.31, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 0.99 ± 1.17, Median: 0.50, IQR: 0.05-1.54, Range: 0.00-4.88
Completeness: 43.40 ± 36.98, Median: 26.90, IQR: 9.20-89.30, Range: 2.40-100.00
Total length: 1.78 ± 1.58, Median: 1.15, IQR: 0.40-2.93, Range: 0.05-7.12

Species-level clusters with contamination < 10%
Total number: 830.0
Cluster members: 1.08 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.40 ± 1.91, Median: 0.61, IQR: 0.06-1.87, Range: 0.00-9.98
Completeness: 44.33 ± 3

# Clusters given Bowers et al / MIMAG classification

## HQ: High-quality species-level clusters (contamination < 5% and completeness > 90%)

In [6]:
hq_df = reps_df.query("Contamination < 5 and Completeness > 90")
hq_df

,MAG,Domain,Phylum,Class,Order,Family,Genus,Species,Cluster members,Completeness,...,kegg_dTDP-D-angolosamine biosynthesis,kegg_dTDP-D-desosamine biosynthesis,kegg_dTDP-D-forosamine biosynthesis,kegg_dTDP-D-mycaminose biosynthesis,kegg_dTDP-L-megosamine biosynthesis,kegg_dTDP-L-mycarose biosynthesis,kegg_dTDP-L-oleandrose biosynthesis,kegg_dTDP-L-olivose biosynthesis,kegg_dTDP-L-rhamnose biosynthesis,kegg_dTDP-beta-L-noviose biosynthesis
0,SRR22878281_binette_bin2,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
1,SRR22878283_binette_bin9,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN
2,SRR22878281_binette_bin11,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
3,SRR22878281_binette_bin11,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
4,SRR22878281_binette_bin12,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,SRR22878281_binette_bin45,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,90.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
186,SRR22878281_binette_bin45,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,90.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
187,SRR22878283_binette_bin36,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,Croceitalea vernalis,2,90.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
188,SRR22878282_binette_bin21,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,2,90.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN


In [7]:
compute_print_stats(hq_df) 

Total number: 182.0
Cluster members: 1.18 ± 0.46, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.19 ± 1.05, Median: 1.02, IQR: 0.27-1.72, Range: 0.00-4.65
Completeness: 96.31 ± 3.25, Median: 96.90, IQR: 93.28-99.60, Range: 90.10-100.00
Total length: 3.76 ± 1.17, Median: 3.55, IQR: 2.96-4.43, Range: 1.03-7.12


### Taxonomy

In [8]:
compute_taxo_classification_summary(hq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,182,0,100.0,0.0
Phylum,182,0,100.0,0.0
Class,182,0,100.0,0.0
Order,182,0,100.0,0.0
Family,182,0,100.0,0.0
Genus,182,0,100.0,0.0
Species,168,14,92.31,7.69


In [9]:
hq_taxo_levels = get_all_taxo_levels(hq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
unclassified,182.0,100.0,215.0
TOTAL,182.0,100.0,215.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
unclassified,182.0,100.0,215.0
TOTAL,182.0,100.0,215.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
unclassified,182.0,100.0,215.0
TOTAL,182.0,100.0,215.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,182.0,100.0,215.0
TOTAL,182.0,100.0,215.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,168.0,92.307692,188.0
Cellulophaga lytica,1.0,0.549451,3.0
Dokdonia sp947496725,1.0,0.549451,3.0
Olleya sediminilitoris,1.0,0.549451,3.0
Pseudoalteromonas marina,1.0,0.549451,3.0
Croceitalea vernalis,1.0,0.549451,2.0
Lacinutrix sp000211855,1.0,0.549451,2.0
Maribacter litoralis,1.0,0.549451,2.0
Marinagarivorans sp947494095,1.0,0.549451,2.0


### Relative abundance

In [10]:
hq_relative_abund_df = get_relative_abund_taxo_levels(hq_df, coverage_df)

Unmapped reads: 80.68 ± 7.93, Median: 77.16, IQR: 76.14-83.46, Range: 75.12-89.76
Mapped reads: 19.32 ± 7.93, Median: 22.84, IQR: 16.54-23.86, Range: 10.24-24.88

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
unclassified,3.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,3.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,78.459246,18.961977,58.060492,69.914462,81.768433,88.658622,95.548812
Cellulophaga lytica,3.0,10.178136,13.866727,0.301126,2.251786,4.202447,15.116640,26.030834
Dokdonia sp947496725,3.0,4.216697,4.807290,1.005900,1.453250,1.900600,5.822095,9.743590
Pseudoalteromonas atlantica,3.0,1.413222,1.257358,0.157823,0.783568,1.409312,2.040922,2.672531
Lacinutrix sp000211855,3.0,1.361587,1.167712,0.129317,0.816515,1.503712,1.977722,2.451731
Maribacter litoralis,3.0,1.104303,1.363866,0.086823,0.329433,0.572042,1.613043,2.654043
Pseudoalteromonas marina,3.0,0.905768,0.924540,0.334394,0.372437,0.410479,1.191454,1.972430
Marinagarivorans sp947494095,3.0,0.423268,0.481044,0.021210,0.156798,0.292387,0.624298,0.956208
Olleya sediminilitoris,3.0,0.417397,0.216793,0.201522,0.308547,0.415572,0.525334,0.635096


### Functions

In [11]:
print_stats(get_bakta_annot_df(hq_df).describe())

CDSs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
CRISPR arrays: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
gaps: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
hypotheticals: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNA regions: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriCs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriTs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriVs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
pseudogenes: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
rRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
sORFs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
signal peptides: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tmRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan


In [12]:
get_kegg_path_df(hq_df)

Before removing rows and columns with only zeros:
Clusters: 182
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 182
KEGG modules: 378

KEGG modules: 194.59 ± 21.95, Median: 195.00, IQR: 186.25-205.00, Range: 78.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate","Aerobactin biosynthesis, lysine => aerobactin",...,beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA","beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis
0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
1,0.0,22.22,0.0,100.0,0.0,0.0,0.0,100.0,100.00,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,50.0
2,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
3,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
4,0.0,27.78,0.0,20.0,0.0,0.0,0.0,100.0,100.00,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,0.0,25.00,0.0,0.0,0.0,0.0,50.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
186,0.0,25.00,0.0,0.0,0.0,0.0,50.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.0
187,0.0,27.78,0.0,0.0,20.0,0.0,0.0,100.0,50.00,0.0,...,100.0,100.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,100.0
188,0.0,24.54,0.0,0.0,20.0,0.0,0.0,100.0,50.00,0.0,...,50.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,50.0


## MQ: Medium-quality species-level clusters (contamination < 10% and completeness > 50%)


In [13]:
mq_df = reps_df.query("Contamination < 10 and Completeness > 50")
compute_print_stats(mq_df) 

Total number: 328.0
Cluster members: 1.16 ± 0.43, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 2.17 ± 2.08, Median: 1.61, IQR: 0.78-2.86, Range: 0.00-9.98
Completeness: 86.59 ± 14.03, Median: 92.25, IQR: 80.15-97.80, Range: 50.60-100.00
Total length: 3.37 ± 1.23, Median: 3.19, IQR: 2.47-4.02, Range: 0.22-7.12


### Taxonomy

In [14]:
compute_taxo_classification_summary(mq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,327,1,99.7,0.3
Phylum,328,0,100.0,0.0
Class,328,0,100.0,0.0
Order,328,0,100.0,0.0
Family,328,0,100.0,0.0
Genus,328,0,100.0,0.0
Species,301,27,91.77,8.23


In [15]:
mq_taxo_levels = get_all_taxo_levels(mq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
unclassified,327.0,99.695122,378.0
Bacteria,1.0,0.304878,2.0
TOTAL,328.0,100.000000,380.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
unclassified,328.0,100.0,380.0
TOTAL,328.0,100.0,380.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
unclassified,328.0,100.0,380.0
TOTAL,328.0,100.0,380.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,328.0,100.0,380.0
TOTAL,328.0,100.0,380.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,301.0,91.768293,337.0
Cellulophaga lytica,1.0,0.304878,3.0
Dokdonia sp947496725,1.0,0.304878,3.0
Pseudoalteromonas marina,1.0,0.304878,3.0
Olleya sediminilitoris,1.0,0.304878,3.0
Polaribacter marinaquae,1.0,0.304878,2.0
Croceitalea vernalis,1.0,0.304878,2.0
Lacinutrix sp000211855,1.0,0.304878,2.0
Maribacter litoralis,1.0,0.304878,2.0


### Relative abundance

In [16]:
mq_relative_abund_df = get_relative_abund_taxo_levels(mq_df, coverage_df)

Unmapped reads: 73.26 ± 8.53, Median: 69.14, IQR: 68.35-76.10, Range: 67.56-83.06
Mapped reads: 26.74 ± 8.53, Median: 30.86, IQR: 23.90-31.65, Range: 16.94-32.44

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
unclassified,3.0,100.0,2.009718e-14,100.0,100.0,100.0,100.0,100.0



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,3.0,100.0,2.009718e-14,100.0,100.0,100.0,100.0,100.0



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,80.385941,13.741677,66.462328,73.609835,80.757341,87.347747,93.938153
Cellulophaga lytica,3.0,7.038323,9.846555,0.242826,1.392279,2.541733,10.436071,18.330409
Dokdonia sp947496725,3.0,2.940641,3.399557,0.811150,0.980338,1.149525,4.005386,6.861247
Vibrio cyclitrophicus,3.0,1.179877,1.696689,0.024332,0.205920,0.387507,1.757650,3.127792
Tenacibaculum sp004337695,3.0,0.938796,0.850838,0.239604,0.465137,0.690670,1.288393,1.886115
Vibrio coralliirubri,3.0,0.916185,1.264004,0.017500,0.193522,0.369544,1.365527,2.361510
Pseudoalteromonas atlantica,3.0,0.912028,0.747816,0.127268,0.559839,0.992410,1.304408,1.616406
Lacinutrix sp000211855,3.0,0.882009,0.706106,0.104280,0.581583,1.058885,1.270873,1.482861
Maribacter litoralis,3.0,0.692686,0.807610,0.070013,0.236417,0.402821,1.004023,1.605224


### Functions

In [17]:
print_stats(get_bakta_annot_df(mq_df).describe())

CDSs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
CRISPR arrays: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
gaps: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
hypotheticals: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNA regions: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriCs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriTs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriVs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
pseudogenes: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
rRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
sORFs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
signal peptides: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tmRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan


In [18]:
get_kegg_path_df(mq_df)

Before removing rows and columns with only zeros:
Clusters: 328
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 328
KEGG modules: 386

KEGG modules: 188.32 ± 30.16, Median: 192.00, IQR: 181.00-201.25, Range: 2.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core","2-Oxocarboxylic acid chain extension, 2-oxoglutarate => 2-oxoadipate => 2-oxopimelate => 2-oxosuberate",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate",...,"beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-D-mycaminose biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-mycarose biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis,dTDP-beta-L-noviose biosynthesis
0,0.0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
1,0.0,0.0,22.22,0.0,100.0,0.0,0.0,0.0,100.0,100.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
2,0.0,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
3,0.0,0.0,11.11,0.0,100.0,0.0,0.0,0.0,100.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
4,0.0,0.0,27.78,0.0,20.0,0.0,0.0,0.0,100.0,100.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0.0,0.0,13.89,0.0,0.0,0.0,0.0,0.0,50.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
326,0.0,0.0,13.89,0.0,0.0,0.0,0.0,0.0,50.0,66.67,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.00,0.0
327,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
328,0.0,0.0,14.81,0.0,0.0,0.0,12.5,0.0,75.0,33.33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.33,0.0


### LQ: Low-quality species-level clusters (contamination < 10% and completeness < 50%)

In [19]:
lq_df = reps_df.query("Contamination < 10 and Completeness < 50")
compute_print_stats(lq_df)

Total number: 502.0
Cluster members: 1.03 ± 0.16, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-2.00
Contamination: 0.89 ± 1.60, Median: 0.17, IQR: 0.01-1.01, Range: 0.00-8.14
Completeness: 16.72 ± 12.34, Median: 11.55, IQR: 7.40-24.30, Range: 0.00-49.40
Total length: 0.80 ± 0.82, Median: 0.54, IQR: 0.29-1.06, Range: 0.05-7.59
